Crypto Price Trend Analysis with PySpark | Stock Market & E-commerce Use Cases 2025

🔍 Want to analyze cryptocurrency price trends and detect profitable patterns? In this video, I’ll show you how to use PySpark to process and analyze historical coin price data to detect trends like consecutive price increases, volume surges, and more! 📊

📌 What You’ll Learn in This Video:

✅ How to create sample crypto price data (Bitcoin, Ethereum, Solana) in PySpark

✅ Use Window Functions to find coins with 3+ days of consecutive price growth

✅ Running optimized queries for trend detection



💡 Where Else Can You Use This Concept?

This method isn’t just for crypto! The same technique is used in:

📈 Stock Market – Find stocks with consecutive price gains

🛒 E-commerce – Identify trending products with increasing sales

🌦️ Weather Data – Detect heatwaves with rising temperatures

🏢 Employee Productivity – Track employees with continuous performance improvements



🚀 Real-World Applications:

🔹 Traders use this analysis to find momentum stocks & crypto trends

🔹 Retailers track bestselling products

🔹 HR teams identify top-performing employees



🛠 Technologies Used:


PySpark for data processing

SQL & Window Functions for trend detection

Pandas for small-scale analysis

📊 Dataset: Historical cryptocurrency prices 📈


In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession
    .builder
    .appName("Spark-Q004")
    .master("local[*]").config("spark.ui.port", "4040")
    .getOrCreate()
)

#
print(spark.sparkContext.uiWebUrl)

#
spark

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/27 11:06:42 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


http://vmware-kubuntu.sandbox.net:4040


In [1]:
data = [
    (1, "Bitcoin", "2024-02-01", 430, 435),
    (1, "Bitcoin", "2024-02-02", 435, 440),
    (1, "Bitcoin", "2024-02-03", 440, 439),
    (1, "Bitcoin", "2024-02-04", 445, 430),
    (2, "Ethereum", "2024-02-01", 320, 325),
    (2, "Ethereum", "2024-02-02", 325, 330),
    (2, "Ethereum", "2024-02-03", 330, 335),
    (2, "Ethereum", "2024-02-04", 335, 340),
    (3, "Solana", "2024-02-01", 120, 125),
    (3, "Solana", "2024-02-02", 125, 130),
    (3, "Solana", "2024-02-03", 130, 135),
    (3, "Solana", "2024-02-04", 135, 140),
]

columns = ["Coin_ID", "Coin_Name", "Date", "Open_Price", "Close_Price"]
df = spark.createDataFrame(data, columns)

#
df.show()

+-------+---------+----------+----------+-----------+
|Coin_ID|Coin_Name|      Date|Open_Price|Close_Price|
+-------+---------+----------+----------+-----------+
|      1|  Bitcoin|2024-02-01|       430|        435|
|      1|  Bitcoin|2024-02-02|       435|        440|
|      1|  Bitcoin|2024-02-03|       440|        439|
|      1|  Bitcoin|2024-02-04|       445|        430|
|      2| Ethereum|2024-02-01|       320|        325|
|      2| Ethereum|2024-02-02|       325|        330|
|      2| Ethereum|2024-02-03|       330|        335|
|      2| Ethereum|2024-02-04|       335|        340|
|      3|   Solana|2024-02-01|       120|        125|
|      3|   Solana|2024-02-02|       125|        130|
|      3|   Solana|2024-02-03|       130|        135|
|      3|   Solana|2024-02-04|       135|        140|
+-------+---------+----------+----------+-----------+



##### Convert Date to proper format

In [4]:
from pyspark.sql.functions import *

df = df.withColumn("Date", to_date(col("Date"), "yyyy-MM-dd"))
df.show()


+-------+---------+----------+----------+-----------+
|Coin_ID|Coin_Name|      Date|Open_Price|Close_Price|
+-------+---------+----------+----------+-----------+
|      1|  Bitcoin|2024-02-01|       430|        435|
|      1|  Bitcoin|2024-02-02|       435|        440|
|      1|  Bitcoin|2024-02-03|       440|        439|
|      1|  Bitcoin|2024-02-04|       445|        430|
|      2| Ethereum|2024-02-01|       320|        325|
|      2| Ethereum|2024-02-02|       325|        330|
|      2| Ethereum|2024-02-03|       330|        335|
|      2| Ethereum|2024-02-04|       335|        340|
|      3|   Solana|2024-02-01|       120|        125|
|      3|   Solana|2024-02-02|       125|        130|
|      3|   Solana|2024-02-03|       130|        135|
|      3|   Solana|2024-02-04|       135|        140|
+-------+---------+----------+----------+-----------+



##### Define Window function to check previous 2 days



In [11]:
from pyspark.sql.window import *
import pyspark.sql.functions as F

windowSpec = Window.partitionBy("Coin_ID").orderBy("Date")
df_with_lag = df.withColumn("Prev_Close_1", F.lag("Close_Price", 1).over(windowSpec)) \
                .withColumn("Prev_Close_2", F.lag("Close_Price", 2).over(windowSpec))

df_with_lag.show()

+-------+---------+----------+----------+-----------+------------+------------+
|Coin_ID|Coin_Name|      Date|Open_Price|Close_Price|Prev_Close_1|Prev_Close_2|
+-------+---------+----------+----------+-----------+------------+------------+
|      1|  Bitcoin|2024-02-01|       430|        435|        NULL|        NULL|
|      1|  Bitcoin|2024-02-02|       435|        440|         435|        NULL|
|      1|  Bitcoin|2024-02-03|       440|        439|         440|         435|
|      1|  Bitcoin|2024-02-04|       445|        430|         439|         440|
|      2| Ethereum|2024-02-01|       320|        325|        NULL|        NULL|
|      2| Ethereum|2024-02-02|       325|        330|         325|        NULL|
|      2| Ethereum|2024-02-03|       330|        335|         330|         325|
|      2| Ethereum|2024-02-04|       335|        340|         335|         330|
|      3|   Solana|2024-02-01|       120|        125|        NULL|        NULL|
|      3|   Solana|2024-02-02|       125

##### Filter coins that had a price increase for 3 consecutive days



In [17]:

df_result = df_with_lag.filter(
    (col("Close_Price") > col("Prev_Close_1")) &
    (col("Prev_Close_1") > col("Prev_Close_2"))
).select("Coin_ID", "Coin_Name").distinct()

df_result.show()

+-------+---------+
|Coin_ID|Coin_Name|
+-------+---------+
|      2| Ethereum|
|      3|   Solana|
+-------+---------+

